# SOC-Graph Pipeline Demo

**End-to-end APT detection on a synthetic provenance graph.**

**Scenario:** A red team exploited an nginx web server (`/usr/sbin/nginx`),
spawned `/bin/sh`, downloaded tooling from C2 at `128.55.12.167`,
and exfiltrated `/etc/passwd` and `/root/.ssh/id_rsa`.
Normal nginx / sshd / python activity runs as background noise.

This notebook runs:
1. Graph construction from raw CSV events
2. Behavioral anomaly detector *(no GPU required)*
3. GNN encoder-decoder *(temporal GAT + GRU — skipped if torch unavailable)*
4. Attack subgraph extraction
5. MITRE ATT\&CK mapping
6. LLM investigation report *(placeholder if no Ollama/OpenAI configured)*


In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

from datetime import timedelta
from soc_graph.data.pidsmaker import load_events
from soc_graph.data.dataset import build_datasets

CSV = "../data/demo/apt_scenario.csv"
events = load_events(CSV)
snap_ds, art_ds = build_datasets(events, window=timedelta(minutes=15))

print(f"Total events  : {len(events)}")
print(f"Time windows  : {len(snap_ds)}")
print()
for i, snap in enumerate(snap_ds.snapshots):
    label = "BENIGN" if i < 3 else "*** ATTACK ***"
    print(
        f"  Window {i}  {snap.window_start.strftime('%H:%M')}-{snap.window_end.strftime('%H:%M')}"
        f"  nodes={len(snap.nodes):2d}  agg-edges={len(snap.edges):2d}"
        f"  raw-obs={snap.total_edge_observations:3d}  [{label}]"
    )


Total events  : 107
Time windows  : 4

  Window 0  08:00-08:15  nodes=15  agg-edges=13  raw-obs= 38  [BENIGN]
  Window 1  08:15-08:30  nodes=16  agg-edges=13  raw-obs= 28  [BENIGN]
  Window 2  08:30-08:45  nodes= 7  agg-edges= 5  raw-obs= 22  [BENIGN]
  Window 3  08:45-09:00  nodes=12  agg-edges=13  raw-obs= 19  [*** ATTACK ***]


## Edge-type distribution per window

In [2]:
for i, snap in enumerate(snap_ds.snapshots):
    label = "benign" if i < 3 else "ATTACK"
    counts = snap.edge_type_counts
    parts = "  ".join(
        f"{k.value}:{v}"
        for k, v in sorted(counts.items(), key=lambda x: -x[1])
    )
    print(f"Window {i} [{label}]  {parts}")


Window 0 [benign]  FORK:6  READ:4  CONNECT:1  SEND:1  WRITE:1
Window 1 [benign]  FORK:8  READ:4  CONNECT:1
Window 2 [benign]  READ:3  CONNECT:1  WRITE:1
Window 3 [ATTACK]  READ:3  CONNECT:3  FORK:2  WRITE:2  EXECUTE:1  SEND:1  RECV:1


## Behavioral anomaly detector (no GPU)

In [3]:
from soc_graph.model.pipeline import run_baseline_experiment

result = run_baseline_experiment(snap_ds, art_ds, benign_ratio=0.75, threshold_k=2.5)
ts = result.training_summary

print(f"Training windows  : {ts.num_windows}")
print(f"Mean edges/window : {ts.mean_edges_per_window:.1f}")
print(f"Learned threshold : {ts.learned_threshold:.4f}  (mean + 2.5 sigma of benign scores)")
print()
print("Flagged edge counts per test window:")
for i, fw in enumerate(result.flagged_windows):
    bar = "=" * min(len(fw), 50)
    print(f"  Test window {i}: {len(fw):3d} flagged  |{bar}|")


Training windows  : 3
Mean edges/window : 10.3
Learned threshold : 3.8143  (mean + 2.5 sigma of benign scores)

Flagged edge counts per test window:
  Test window 0:  12 flagged  |============|


## Flagged attack subgraph

In [4]:
from soc_graph.detection.subgraph import build_reduced_subgraph
from soc_graph.report.serialize import serialize_reduced_subgraph

alerts = []
for i, (snap, flagged) in enumerate(zip(result.test_snapshots, result.flagged_windows)):
    if not flagged:
        continue
    aid = f"behavioral-alert-{i+1:03d}"
    reduced = build_reduced_subgraph(snap, flagged, aid)
    payload = serialize_reduced_subgraph(reduced)
    alerts.append(payload)

    print(f"Alert: {aid}")
    print(f"  Nodes in subgraph : {len(payload['nodes'])}")
    print(f"  Flagged edges     : {payload['flagged_edge_count']} / {payload['total_edge_count']} total")
    print(f"  Components        : {payload['component_count']}")
    print()
    print("  Nodes:")
    for n in payload["nodes"]:
        print(f"    [{n['type']:7s}]  {n['name']}")
    print()
    print("  Top anomalous edges:")
    top = sorted(payload["edges"], key=lambda e: e.get("anomaly_score", 0), reverse=True)[:10]
    for e in top:
        print(
            f"    {e['src_type']:7s} --{e['type']:8s}--> {e['dst_type']:7s}"
            f"  score={e.get('anomaly_score', 0):.3f}  x{e['count']}"
        )


Alert: behavioral-alert-001
  Nodes in subgraph : 11
  Flagged edges     : 12 / 13 total
  Components        : 1

  Nodes:
    [FILE   ]  /tmp/.hidden/payload.sh
    [FILE   ]  /etc/passwd
    [FILE   ]  /root/.ssh/id_rsa
    [FILE   ]  /tmp/.hidden/exfil_tool
    [PROCESS]  /usr/sbin/nginx
    [PROCESS]  /bin/sh
    [PROCESS]  /tmp/.hidden/exfil_tool
    [PROCESS]  /usr/bin/wget
    [SOCKET ]  128.55.12.167:4444
    [SOCKET ]  128.55.12.167:443
    [SOCKET ]  128.55.12.167:443

  Top anomalous edges:
    PROCESS --SEND    --> SOCKET   score=7.850  x2
    FILE    --EXECUTE --> PROCESS  score=7.543  x1
    SOCKET  --RECV    --> PROCESS  score=7.543  x1
    PROCESS --WRITE   --> FILE     score=7.445  x2
    PROCESS --WRITE   --> FILE     score=6.445  x1
    PROCESS --CONNECT --> SOCKET   score=6.157  x1
    PROCESS --CONNECT --> SOCKET   score=6.157  x1
    PROCESS --CONNECT --> SOCKET   score=6.157  x1
    PROCESS --READ    --> FILE     score=5.058  x1
    PROCESS --READ    --> FILE    

## Attack subgraph visualisation

In [5]:
if not alerts:
    print("No alerts to visualise.")
else:
    payload = alerts[0]

    def safe(s):
        for ch in "-:/.":
            s = s.replace(ch, "_")
        return s

    shapes = {"PROCESS": "ellipse", "FILE": "box", "SOCKET": "diamond"}
    colors = {"PROCESS": "#AED6F1", "FILE": "#A9DFBF", "SOCKET": "#F9E79F"}

    lines = [
        "digraph G {",
        '  rankdir="LR";',
        '  node [fontname="Helvetica" fontsize=10];',
        '  edge [fontsize=9];',
    ]
    for n in payload["nodes"]:
        nid = safe(n["id"])
        label = n["name"]
        lines.append(
            f'  {nid} [label="{label}" shape={shapes[n["type"]]} '
            f'style=filled fillcolor="{colors[n["type"]]}"];'
        )
    for e in payload["edges"]:
        src, dst = safe(e["src"]), safe(e["dst"])
        score = e.get("anomaly_score", 0)
        lines.append(
            f'  {src} -> {dst} [label="{e["type"]} x{e["count"]} score={score:.2f}"];'
        )
    lines.append("}")
    dot_src = "\n".join(lines)

    try:
        import graphviz
        display(graphviz.Source(dot_src))
    except Exception:
        print("graphviz not installed — DOT source:")
        print(dot_src)


graphviz not installed — DOT source:
digraph G {
  rankdir="LR";
  node [fontname="Helvetica" fontsize=10];
  edge [fontsize=9];
  file_pay [label="/tmp/.hidden/payload.sh" shape=box style=filled fillcolor="#A9DFBF"];
  file_sens [label="/etc/passwd" shape=box style=filled fillcolor="#A9DFBF"];
  file_ssh [label="/root/.ssh/id_rsa" shape=box style=filled fillcolor="#A9DFBF"];
  file_tool [label="/tmp/.hidden/exfil_tool" shape=box style=filled fillcolor="#A9DFBF"];
  proc_nginx [label="/usr/sbin/nginx" shape=ellipse style=filled fillcolor="#AED6F1"];
  proc_sh [label="/bin/sh" shape=ellipse style=filled fillcolor="#AED6F1"];
  proc_tool [label="/tmp/.hidden/exfil_tool" shape=ellipse style=filled fillcolor="#AED6F1"];
  proc_wget [label="/usr/bin/wget" shape=ellipse style=filled fillcolor="#AED6F1"];
  sock_c2 [label="128.55.12.167:4444" shape=diamond style=filled fillcolor="#F9E79F"];
  sock_c2b [label="128.55.12.167:443" shape=diamond style=filled fillcolor="#F9E79F"];
  sock_exfil [la

## GNN encoder-decoder (temporal GAT + GRU)

In [6]:
from soc_graph.model.runtime import check_torch_backend

backend = check_torch_backend()
print("Torch backend:", "available" if backend.available else f"UNAVAILABLE: {backend.detail}")


Torch backend: available


In [7]:
if backend.available:
    from soc_graph.model.pipeline import run_gnn_experiment

    gnn_result = run_gnn_experiment(
        snap_ds, art_ds,
        benign_ratio=0.75,
        epochs=15,
        threshold_k=2.5,
        checkpoint_path="../artifacts/models/gnn_demo.pt",
    )
    print(f"Final training loss : {gnn_result.final_loss:.4f}")
    print(f"Learned threshold   : {gnn_result.learned_threshold:.4f}")
    print()
    print("Flagged edge counts per test window (GNN):")
    for i, fw in enumerate(gnn_result.flagged_windows):
        bar = "=" * min(len(fw), 50)
        print(f"  Test window {i}: {len(fw):3d} flagged  |{bar}|")

    # Build GNN alerts too
    gnn_alerts = []
    for i, (snap, flagged) in enumerate(zip(gnn_result.test_snapshots, gnn_result.flagged_windows)):
        if not flagged:
            continue
        aid = f"gnn-alert-{i+1:03d}"
        reduced = build_reduced_subgraph(snap, flagged, aid)
        gnn_alerts.append(serialize_reduced_subgraph(reduced))
    print(f"\nGNN raised {len(gnn_alerts)} alert(s).")
else:
    gnn_alerts = []
    print("Skipping GNN — using behavioral alerts for remaining cells.")


Final training loss : 1.5244
Learned threshold   : 0.8575

Flagged edge counts per test window (GNN):
  Test window 0:   0 flagged  ||

GNN raised 0 alert(s).


## MITRE ATT&CK mapping

In [8]:
from soc_graph.report.mitre_mapping import map_subgraph

source_alerts = gnn_alerts if (backend.available and gnn_alerts) else alerts

if source_alerts:
    edge_hints = [
        {
            "src_type": e["src_type"],
            "edge_type": e["type"],
            "dst_type": e["dst_type"],
        }
        for e in source_alerts[0]["edges"]
    ]
    techniques = map_subgraph(edge_hints)
    print(f"Mapped {len(techniques)} ATT&CK techniques from the flagged subgraph:\n")
    for t in techniques:
        print(f"  {t['technique_id']}  {t['technique_name']:<45s}  [{t['tactic']}]")
        print(f"           {t['rationale']}")
        print()


Mapped 14 ATT&CK techniques from the flagged subgraph:

  T1204  User Execution                                 [Execution]
           A file was executed, spawning a process.

  T1059  Command and Scripting Interpreter              [Execution]
           Script or binary executed from file system.

  T1106  Native API                                     [Execution]
           Direct syscall used to create a new process.

  T1074  Data Staged                                    [Collection]
           Process wrote data to a file, possibly staging for exfiltration.

  T1027  Obfuscated Files or Information                [Defense Evasion]
           Written file may be an obfuscated payload or dropped tool.

  T1547  Boot or Logon Autostart Execution              [Persistence]
           Write to startup/config location could establish persistence.

  T1071  Application Layer Protocol                     [Command and Control]
           Process opened a network socket, possibly for C2 c

## Investigation report

In [9]:
from soc_graph.report.llm_report import generate_report, config_from_env

llm_cfg = config_from_env()
if llm_cfg:
    print(f"LLM provider : {llm_cfg.provider} / {llm_cfg.model}")
else:
    print("No LLM configured — generating placeholder report.")
    print("  To enable: ollama pull qwen2.5 && ollama serve")
    print("  Or set: OPENAI_API_KEY=sk-...")
print()

source_alerts = gnn_alerts if (backend.available and gnn_alerts) else alerts

if source_alerts:
    report = generate_report(source_alerts[0], config=llm_cfg)

    icons = {"malicious": "[MALICIOUS]", "suspicious": "[SUSPICIOUS]", "benign": "[BENIGN]"}
    print(f"{icons.get(report.verdict, '[?]')} VERDICT    : {report.verdict.upper()}")
    print(f"             CONFIDENCE : {report.confidence}")
    print()
    print("NARRATIVE:")
    print(f"  {report.narrative}")
    if report.recommended_actions:
        print()
        print("RECOMMENDED ACTIONS:")
        for a in report.recommended_actions:
            print(f"  - {a}")
    if report.note:
        print()
        print(f"Note: {report.note}")


No LLM configured — generating placeholder report.
  To enable: ollama pull qwen2.5 && ollama serve
  Or set: OPENAI_API_KEY=sk-...



[SUSPICIOUS] VERDICT    : SUSPICIOUS
             CONFIDENCE : low

NARRATIVE:
  Alert behavioral-alert-001 contains 12 anomalous aggregated edge(s) whose behaviour deviated significantly from the benign training baseline. Manual analyst review is recommended.

Note: LLM explainer not configured. Set OLLAMA_BASE_URL or OPENAI_API_KEY to enable.


## Status summary

| Component | Status |
|-----------|--------|
| PIDSMaker CSV ingestion | working |
| Time-windowed graph construction | working |
| `GraphTensorArtifact` (PyG-ready tensors) | working |
| Behavioral anomaly detector | working (no GPU) |
| GNN encoder (2-layer GAT + GRU memory) | working (requires torch) |
| GNN edge decoder (MLP) | working |
| Threshold calibration (sigma-based) | working |
| Connected-component subgraph extraction | working |
| MITRE ATT&CK rule mapping | working |
| LLM report generation | wired — needs Ollama or OPENAI_API_KEY |
| FastAPI REST API | implemented |
| Streamlit dashboard | implemented |

**Run the API:**
```bash
SOC_GRAPH_DATA_CSV=data/demo/apt_scenario.csv uvicorn soc_graph.api.app:app --reload
```

**Run the dashboard:**
```bash
streamlit run src/soc_graph/dashboard/streamlit_app.py
```
